# Лабораторная работа №3. Изображения и видео

Задача относится к **детекции объектов**: нужно восстановить bounding box-разметку из `output.mp4` (кадры с нарисованными рамками) и перенести координаты на `input.mp4`. Пайплайн включает декодирование видео в JPEG-кадры, классическое извлечение bbox средствами OpenCV, сравнение с эталонной разметкой CVAT XML по метрикам IoU / precision / recall / F1, экспорт в COCO JSON и дальнейшее обучение детектора Faster R-CNN.

Исходные данные: `data/input.mp4`, `data/output.mp4`, `data/annotations.xml`.


## Установка и импорт библиотек

В коде используются OpenCV, PyTorch, torchvision и torchmetrics. Если ноутбук запускается в чистой среде, первая ячейка установит недостающие зависимости.

In [ ]:
import sys
import subprocess
import importlib.util

packages = {
    "cv2": "opencv-python",
    "torch": "torch",
    "torchvision": "torchvision",
    "torchmetrics": "torchmetrics",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "gdown": "gdown"
}

missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [ ]:
from pathlib import Path
import json
import math
import random
import shutil
import xml.etree.ElementTree as ET
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

## Конфигурация проекта

Исходные видео должны лежать в `data`. Если `annotations.xml` отсутствует, он скачивается по ссылке из задания.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FRAMES_DIR = DATA_DIR / "frames"
OUTPUT_FRAMES_DIR = DATA_DIR / "output_frames"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [DATA_DIR, FRAMES_DIR, OUTPUT_FRAMES_DIR, ARTIFACTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_VIDEO = DATA_DIR / "input.mp4"
OUTPUT_VIDEO = DATA_DIR / "output.mp4"
GT_XML = DATA_DIR / "annotations.xml"

ANNOTATION_FILE_ID = "1QqWoLGIOdXi9ZF2a9ArDJnN94yz1ZUhY"
RANDOM_SEED = 42
FRAME_STRIDE = 1
MIN_BOX_AREA = 80
IOU_MATCH_THRESHOLD = 0.5
EPOCHS = 5
BATCH_SIZE = 2
PATIENCE = 2

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)

In [ ]:
if not GT_XML.exists():
    import gdown
    gdown.download(id=ANNOTATION_FILE_ID, output=str(GT_XML), quiet=False)

if not INPUT_VIDEO.exists() or not OUTPUT_VIDEO.exists():
    raise FileNotFoundError("Положите input.mp4 и output.mp4 в папку data перед запуском ноутбука.")

## Декодирование видео в последовательность кадров

Видео декодируется в растровые JPEG-кадры фиксированного разрешения. Для `input.mp4` сохраняются чистые кадры, для `output.mp4` — кадры с чужой визуализацией bbox. Нумерация синхронизирована: один и тот же `frame_id` соответствует одному моменту времени в обоих роликах.

Формат кадров — **JPEG** (сжатие с потерями). Для задачи извлечения нарисованных рамок это допустимо, так как нас интересуют крупные контрастные изменения, а не субпиксельная точность RAW/TIFF.


In [ ]:
def decode_video(video_path, frames_dir, prefix, stride=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть видео: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    paths = []
    frame_index = 0
    saved_index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_index % stride == 0:
            path = frames_dir / f"{prefix}_{saved_index:06d}.jpg"
            cv2.imwrite(str(path), frame)
            paths.append(path)
            saved_index += 1
        frame_index += 1
    cap.release()
    return {"fps": fps, "width": width, "height": height, "total": total, "paths": paths}

for directory in [FRAMES_DIR, OUTPUT_FRAMES_DIR]:
    for path in directory.glob("*.jpg"):
        path.unlink()

input_meta = decode_video(INPUT_VIDEO, FRAMES_DIR, "frame", FRAME_STRIDE)
output_meta = decode_video(OUTPUT_VIDEO, OUTPUT_FRAMES_DIR, "frame", FRAME_STRIDE)

video_info = pd.DataFrame([
    {"video": "input", "fps": input_meta["fps"], "width": input_meta["width"], "height": input_meta["height"], "frames_original": input_meta["total"], "frames_saved": len(input_meta["paths"])},
    {"video": "output", "fps": output_meta["fps"], "width": output_meta["width"], "height": output_meta["height"], "frames_original": output_meta["total"], "frames_saved": len(output_meta["paths"])}
])
video_info

## Описание видеоданных и пригодность для CV-задачи

Перед извлечением разметки фиксируются параметры исходных роликов и базовые характеристики кадров. Для задачи **детекции объектов** важны:

- стабильное разрешение и FPS у пары `input` / `output`;
- достаточное число кадров;
- фиксированный ракурс камеры (типичная дорожная сцена);
- наличие эталонной разметки в XML для контроля качества.

JPEG используется как компромисс между качеством и объёмом: для поиска нарисованных рамок потерь сжатия обычно достаточно.


In [ ]:
frame_stats = []
for frame_id, (inp, outp) in enumerate(zip(input_meta["paths"], output_meta["paths"])):
    input_frame = cv2.imread(str(inp))
    output_frame = cv2.imread(str(outp))
    diff = cv2.absdiff(input_frame, output_frame)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    frame_stats.append({
        "frame_id": frame_id,
        "input_brightness": float(input_frame.mean()),
        "diff_pixels_pct": float((gray > 25).mean() * 100),
    })
frame_stats_df = pd.DataFrame(frame_stats)

display(video_info)
display(pd.DataFrame({
    "параметр": [
        "формат кадров",
        "разрешение",
        "число синхронных кадров",
        "stride",
        "средняя доля изменённых пикселей (diff)",
    ],
    "значение": [
        "JPEG",
        f"{input_meta['width']}x{input_meta['height']}",
        len(input_meta["paths"]),
        FRAME_STRIDE,
        f"{frame_stats_df['diff_pixels_pct'].mean():.3f}%",
    ],
}))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(frame_stats_df["frame_id"], frame_stats_df["input_brightness"], linewidth=1)
axes[0].set_ylabel("яркость input")
axes[0].set_title("Освещённость и сигнал разметки по времени")
axes[0].grid(alpha=0.3)
axes[1].plot(frame_stats_df["frame_id"], frame_stats_df["diff_pixels_pct"], linewidth=1, color="#d62728")
axes[1].set_xlabel("frame_id")
axes[1].set_ylabel("% diff > 25")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Чтение оригинальной разметки

Эталонная разметка хранится в **CVAT XML** — формат, близкий к PASCAL VOC: для каждого объекта заданы прямоугольник и класс. В нашем файле используется видеоразметка через теги `track` / `box`.

GT применяется только для **оценки качества извлечения**, а не для подмены координат при «воровстве» разметки.


In [ ]:
def parse_cvat_xml(xml_path, frame_width, frame_height):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    image_nodes = root.findall(".//image")
    if image_nodes:
        for image in image_nodes:
            frame_id = int(image.attrib.get("id", len(rows)))
            file_name = image.attrib.get("name", f"frame_{frame_id:06d}.jpg")
            for box in image.findall("box"):
                label = box.attrib.get("label", "object")
                xtl = float(box.attrib["xtl"])
                ytl = float(box.attrib["ytl"])
                xbr = float(box.attrib["xbr"])
                ybr = float(box.attrib["ybr"])
                rows.append({"frame_id": frame_id, "file_name": file_name, "label": label, "x1": xtl, "y1": ytl, "x2": xbr, "y2": ybr})
    for track in root.findall(".//track"):
        label = track.attrib.get("label", "object")
        track_id = int(track.attrib.get("id", -1))
        for box in track.findall("box"):
            if int(box.attrib.get("outside", 0)) == 1:
                continue
            frame_id = int(box.attrib["frame"])
            xtl = float(box.attrib["xtl"])
            ytl = float(box.attrib["ytl"])
            xbr = float(box.attrib["xbr"])
            ybr = float(box.attrib["ybr"])
            rows.append({"frame_id": frame_id, "file_name": f"frame_{frame_id:06d}.jpg", "label": label, "track_id": track_id, "x1": xtl, "y1": ytl, "x2": xbr, "y2": ybr})
    gt = pd.DataFrame(rows)
    if gt.empty:
        raise ValueError("В XML не найдены bbox.")
    gt["x1"] = gt["x1"].clip(0, frame_width - 1)
    gt["y1"] = gt["y1"].clip(0, frame_height - 1)
    gt["x2"] = gt["x2"].clip(0, frame_width - 1)
    gt["y2"] = gt["y2"].clip(0, frame_height - 1)
    gt = gt[gt["x2"].gt(gt["x1"]) & gt["y2"].gt(gt["y1"])].copy()
    return gt.reset_index(drop=True)

NUM_FRAMES = len(input_meta["paths"])
gt_df = parse_cvat_xml(GT_XML, input_meta["width"], input_meta["height"])
gt_eval_df = gt_df[gt_df["frame_id"].lt(NUM_FRAMES)].copy().reset_index(drop=True)
gt_df.head(), gt_eval_df.shape, gt_eval_df["label"].value_counts()


## EDA эталонной разметки и оценка пригодности датасета

Анализ GT отвечает на вопросы пригодности данных для обучения и оценки:

- **объём**: сколько кадров и bbox доступно;
- **баланс классов**: нет ли сильного перекоса;
- **плотность сцены**: сколько объектов в кадре;
- **геометрия bbox**: типичные размеры и положение объектов;
- **согласованность**: все ли `frame_id` из XML соответствуют декодированным кадрам.

Pixel accuracy здесь не используется — для детекции важнее IoU и F1 на уровне объектов.


In [ ]:
gt_eval_df["width"] = gt_eval_df["x2"] - gt_eval_df["x1"]
gt_eval_df["height"] = gt_eval_df["y2"] - gt_eval_df["y1"]
gt_eval_df["area"] = gt_eval_df["width"] * gt_eval_df["height"]
gt_eval_df["cx"] = (gt_eval_df["x1"] + gt_eval_df["x2"]) / 2
gt_eval_df["cy"] = (gt_eval_df["y1"] + gt_eval_df["y2"]) / 2

boxes_per_frame = gt_eval_df.groupby("frame_id").size().rename("gt_boxes")
invalid_gt_frames = gt_df.loc[gt_df["frame_id"].ge(NUM_FRAMES), "frame_id"].nunique()

dataset_overview = pd.DataFrame({
    "метрика": [
        "bbox в GT (валидные кадры)",
        "уникальных кадров с GT",
        "bbox на кадр (median)",
        "bbox на кадр (max)",
        "frame_id в XML без JPEG",
        "классов",
    ],
    "значение": [
        len(gt_eval_df),
        gt_eval_df["frame_id"].nunique(),
        boxes_per_frame.median(),
        boxes_per_frame.max(),
        invalid_gt_frames,
        gt_eval_df["label"].nunique(),
    ],
})
display(dataset_overview)
display(gt_eval_df["label"].value_counts().to_frame("count"))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
gt_eval_df["label"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#1f77b4", "#ff7f0e"])
axes[0, 0].set_title("Баланс классов GT")
axes[0, 0].set_xlabel("класс")
boxes_per_frame.plot(ax=axes[0, 1], linewidth=1)
axes[0, 1].set_title("Число GT bbox на кадр")
axes[0, 1].set_xlabel("frame_id")
gt_eval_df["area"].plot(kind="hist", bins=30, ax=axes[1, 0], color="#2ca02c")
axes[1, 0].set_title("Распределение площади bbox")
axes[1, 0].set_xlabel("area, px")
axes[1, 1].hexbin(gt_eval_df["cx"], gt_eval_df["cy"], gridsize=30, cmap="YlOrRd")
axes[1, 1].set_title("Heatmap центров bbox")
axes[1, 1].set_xlabel("cx")
axes[1, 1].set_ylabel("cy")
plt.tight_layout()
plt.show()


## Метрики IoU и сопоставление объектов

Для детекции используются метрики на уровне bbox, а не pixel accuracy (она игнорирует дисбаланс «фон vs объект»).

- **IoU (Jaccard index)** — пересечение / объединение двух прямоугольников; порог 0.5 считается стандартным для «совпадения».
- **Precision** — доля извлечённых bbox, которые действительно совпали с GT.
- **Recall** — доля GT bbox, которые удалось восстановить.
- **F1** — баланс precision и recall.

Сопоставление выполняется жадно по убыванию IoU отдельно для каждого кадра.


In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0

def evaluate_boxes(gt, pred, threshold=0.5):
    rows = []
    total_tp = 0
    total_fp = 0
    total_fn = 0
    matched_ious = []
    valid_gt = gt[gt["frame_id"].lt(NUM_FRAMES)]
    frame_ids = sorted(set(valid_gt["frame_id"].unique()).union(set(pred["frame_id"].unique())))
    for frame_id in frame_ids:
        g = valid_gt[valid_gt["frame_id"].eq(frame_id)].reset_index(drop=True)
        p = pred[pred["frame_id"].eq(frame_id)].reset_index(drop=True)
        pairs = []
        for gi, gr in g.iterrows():
            for pi, pr in p.iterrows():
                iou = box_iou([gr.x1, gr.y1, gr.x2, gr.y2], [pr.x1, pr.y1, pr.x2, pr.y2])
                pairs.append((iou, gi, pi))
        pairs.sort(reverse=True)
        used_g = set()
        used_p = set()
        frame_ious = []
        for iou, gi, pi in pairs:
            if iou < threshold:
                break
            if gi not in used_g and pi not in used_p:
                used_g.add(gi)
                used_p.add(pi)
                frame_ious.append(iou)
        tp = len(frame_ious)
        fp = len(p) - tp
        fn = len(g) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn
        matched_ious.extend(frame_ious)
        rows.append({"frame_id": frame_id, "tp": tp, "fp": fp, "fn": fn, "gt_count": len(g), "pred_count": len(p), "mean_iou": np.mean(frame_ious) if frame_ious else 0})
    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    summary = {"tp": total_tp, "fp": total_fp, "fn": total_fn, "precision": precision, "recall": recall, "f1": f1, "mean_iou": float(np.mean(matched_ious)) if matched_ious else 0, "matches": len(matched_ious)}
    return pd.DataFrame(rows), summary, matched_ious


## Метод 1: разность `output` и `input` (пороговая обработка)

Классический pipeline CV: **absdiff** → grayscale → **бинаризация** → **морфологическое closing** → поиск контуров. Метод напрямую использует тот факт, что разметка на `output.mp4` — это новые пиксели поверх исходного кадра.


In [ ]:
def merge_rectangles(rects, distance=10):
    rects = [list(map(float, r)) for r in rects]
    changed = True
    while changed:
        changed = False
        merged = []
        used = [False] * len(rects)
        for i, a in enumerate(rects):
            if used[i]:
                continue
            x1, y1, x2, y2 = a
            used[i] = True
            for j, b in enumerate(rects):
                if used[j]:
                    continue
                bx1, by1, bx2, by2 = b
                separated = x2 + distance < bx1 or bx2 + distance < x1 or y2 + distance < by1 or by2 + distance < y1
                if not separated:
                    x1, y1, x2, y2 = min(x1, bx1), min(y1, by1), max(x2, bx2), max(y2, by2)
                    used[j] = True
                    changed = True
            merged.append([x1, y1, x2, y2])
        rects = merged
    return rects

def boxes_from_diff(input_frame, output_frame):
    diff = cv2.absdiff(input_frame, output_frame)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w * h >= MIN_BOX_AREA and w > 5 and h > 5:
            rects.append([x, y, x + w, y + h])
    return merge_rectangles(rects, distance=12)

## Метод 2: цветовая сегментация bbox (HSV)

Если рамки нарисованы насыщенным цветом, их можно выделить в **HSV**-пространстве (`inRange`) и дополнительно пересечь с границами Canny. Это пример **пороговой сегментации по цвету**.


In [ ]:
def boxes_from_hsv(output_frame):
    hsv = cv2.cvtColor(output_frame, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 60, 80])
    upper = np.array([179, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    gray = cv2.cvtColor(output_frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    mask = cv2.bitwise_and(mask, edges)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w * h >= MIN_BOX_AREA and w > 5 and h > 5:
            rects.append([x, y, x + w, y + h])
    return merge_rectangles(rects, distance=16)

## Метод 3: контуры Canny на `output.mp4`

Метод опирается на **выделение границ** и аппроксимацию контуров прямоугольниками. Не использует `input.mp4`, поэтому более чувствителен к фону и шуму на output-кадре.


In [ ]:
def boxes_from_edges(output_frame):
    gray = cv2.cvtColor(output_frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 80, 180)
    kernel = np.ones((3, 3), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rects = []
    h_img, w_img = gray.shape
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        if area >= MIN_BOX_AREA and w > 8 and h > 8 and area < 0.8 * w_img * h_img:
            approx = cv2.approxPolyDP(contour, 0.03 * cv2.arcLength(contour, True), True)
            if len(approx) >= 4:
                rects.append([x, y, x + w, y + h])
    return merge_rectangles(rects, distance=10)

## Запуск методов извлечения bbox

Для каждого синхронного кадра применяются три классических метода OpenCV. Результаты сравниваются с GT по IoU; лучший метод выбирается автоматически по F1 и mean IoU.


In [ ]:
def extract_predictions(method_name):
    rows = []
    for frame_id, (input_path, output_path) in enumerate(tqdm(list(zip(input_meta["paths"], output_meta["paths"])), desc=method_name)):
        input_frame = cv2.imread(str(input_path))
        output_frame = cv2.imread(str(output_path))
        if method_name == "diff":
            boxes = boxes_from_diff(input_frame, output_frame)
        elif method_name == "hsv":
            boxes = boxes_from_hsv(output_frame)
        elif method_name == "edges":
            boxes = boxes_from_edges(output_frame)
        else:
            raise ValueError(method_name)
        for box in boxes:
            x1, y1, x2, y2 = box
            rows.append({"frame_id": frame_id, "label": "object", "x1": x1, "y1": y1, "x2": x2, "y2": y2, "score": 1.0, "method": method_name})
    return pd.DataFrame(rows)

predictions = {method: extract_predictions(method) for method in ["diff", "hsv", "edges"]}
comparison = []
frame_metrics = {}
matched_iou_by_method = {}
for method, pred in predictions.items():
    per_frame, summary, matched_ious = evaluate_boxes(gt_eval_df, pred, IOU_MATCH_THRESHOLD)
    summary["method"] = method
    summary["boxes"] = len(pred)
    comparison.append(summary)
    frame_metrics[method] = per_frame
    matched_iou_by_method[method] = matched_ious

comparison_df = pd.DataFrame(comparison).sort_values(["f1", "mean_iou"], ascending=False).reset_index(drop=True)
comparison_df


In [ ]:
best_method = comparison_df.iloc[0]["method"]
best_pred_df = predictions[best_method].copy().reset_index(drop=True)
best_method

## Сравнение методов извлечения bbox

Ниже — сводные метрики трёх классических подходов OpenCV и их покадровый разбор. Метод **diff** обычно выигрывает, потому что разметка на `output.mp4` — это добавленные поверх `input.mp4` пиксели; пороговая обработка разности напрямую выделяет эти изменения.

Дополнительно анализируются типы ошибок (TP / FP / FN) и распределение IoU у успешно сопоставленных пар bbox.


In [ ]:
metrics = ["precision", "recall", "f1", "mean_iou"]
x = np.arange(len(comparison_df))
width = 0.18
fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(metrics):
    ax.bar(x + (i - 1.5) * width, comparison_df[metric], width=width, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["method"])
ax.set_ylim(0, 1.05)
ax.set_title("Сводные метрики методов извлечения")
ax.set_xlabel("Метод OpenCV")
ax.set_ylabel("Значение")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(comparison_df))
for col, color in [("tp", "#2ca02c"), ("fp", "#ff7f0e"), ("fn", "#d62728")]:
    ax.bar(comparison_df["method"], comparison_df[col], bottom=bottom, label=col.upper(), color=color)
    bottom += comparison_df[col].values
ax.set_title("Структура ошибок по методам")
ax.set_xlabel("Метод")
ax.set_ylabel("Количество bbox")
ax.legend()
plt.tight_layout()
plt.show()

best_fm = frame_metrics[comparison_df.iloc[0]["method"]].sort_values("frame_id")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(best_fm["frame_id"], best_fm["mean_iou"], linewidth=1)
ax.set_title(f"Mean IoU по кадрам ({comparison_df.iloc[0]['method']})")
ax.set_xlabel("frame_id")
ax.set_ylabel("mean IoU")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for method, ious in matched_iou_by_method.items():
    if ious:
        ax.hist(ious, bins=20, alpha=0.5, label=method)
ax.set_title("Распределение IoU у matched bbox")
ax.set_xlabel("IoU")
ax.set_ylabel("Число пар")
ax.legend()
plt.tight_layout()
plt.show()


## Присвоение классов извлеченным bbox

OpenCV-методы восстанавливают **координаты**, но не класс объекта. Для COCO и обучения детектора класс назначается постфактум: каждому extracted bbox ставится label GT bbox с максимальным IoU на том же кадре.

Это отдельный этап анализа: важно понимать, какая доля bbox получила надёжное имя класса (IoU ≥ порога), а какая осталась условным `object`.


In [ ]:
def assign_labels_from_gt(pred, gt):
    labeled = pred.copy()
    labels = []
    match_ious = []
    for _, pr in labeled.iterrows():
        g = gt[gt["frame_id"].eq(pr.frame_id)]
        best_label = "object"
        best_iou = 0.0
        for _, gr in g.iterrows():
            iou = box_iou([gr.x1, gr.y1, gr.x2, gr.y2], [pr.x1, pr.y1, pr.x2, pr.y2])
            if iou > best_iou:
                best_iou = iou
                best_label = gr.label
        labels.append(best_label)
        match_ious.append(best_iou)
    labeled["label"] = labels
    labeled["match_iou"] = match_ious
    return labeled

best_pred_df = assign_labels_from_gt(best_pred_df, gt_eval_df)
label_quality = pd.DataFrame({
    "всего bbox": [len(best_pred_df)],
    "IoU >= 0.5": [(best_pred_df["match_iou"] >= IOU_MATCH_THRESHOLD).sum()],
    "IoU < 0.5": [(best_pred_df["match_iou"] < IOU_MATCH_THRESHOLD).sum()],
    "средний match IoU": [best_pred_df["match_iou"].mean()],
})
label_quality


## Качество присвоения классов extracted bbox

Графики показывают, насколько надёжно extracted bbox сопоставляются с GT не только по координатам, но и по названию класса.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
best_pred_df["label"].value_counts().plot(kind="bar", ax=axes[0], color="#9467bd")
axes[0].set_title("Классы после assign_labels_from_gt")
axes[0].set_xlabel("класс")
best_pred_df["match_iou"].plot(kind="hist", bins=25, ax=axes[1], color="#8c564b")
axes[1].axvline(IOU_MATCH_THRESHOLD, color="red", linestyle="--", label=f"threshold={IOU_MATCH_THRESHOLD}")
axes[1].set_title("Распределение match IoU")
axes[1].set_xlabel("match IoU")
axes[1].legend()
plt.tight_layout()
plt.show()

pd.crosstab(
    best_pred_df["label"],
    pd.cut(best_pred_df["match_iou"], bins=[0, 0.5, 1.0], labels=["<0.5", ">=0.5"], include_lowest=True),
)


## Формирование COCO-аннотаций

Извлечённые bbox экспортируются в **COCO JSON** — стандартный формат для обучения детекторов (`images`, `annotations`, `categories`). Координаты bbox записываются в формате `[x, y, width, height]`; файлы изображений остаются в `data/frames`.


In [ ]:
def dataframe_to_coco(df, image_paths, width, height, output_path):
    labels = sorted(df["label"].dropna().unique().tolist())
    categories = [{"id": i + 1, "name": label, "supercategory": "object"} for i, label in enumerate(labels)]
    label_to_id = {item["name"]: item["id"] for item in categories}
    images = []
    for image_id, path in enumerate(image_paths):
        images.append({"id": image_id, "file_name": str(Path("data") / "frames" / path.name), "width": width, "height": height})
    annotations = []
    ann_id = 1
    for _, row in df.iterrows():
        x = float(row.x1)
        y = float(row.y1)
        w = float(row.x2 - row.x1)
        h = float(row.y2 - row.y1)
        if w <= 0 or h <= 0:
            continue
        annotations.append({"id": ann_id, "image_id": int(row.frame_id), "category_id": int(label_to_id[row.label]), "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
        ann_id += 1
    coco = {"images": images, "annotations": annotations, "categories": categories}
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)
    return coco

COCO_PATH = ARTIFACTS_DIR / "annotations_extracted_coco.json"
coco = dataframe_to_coco(best_pred_df, input_meta["paths"], input_meta["width"], input_meta["height"], COCO_PATH)

pd.DataFrame({"images": [len(coco["images"])], "annotations": [len(coco["annotations"])], "categories": [len(coco["categories"])]})

## Проверка COCO-экспорта и баланса классов

Сравнивается эталонная разметка и extracted COCO: число изображений, аннотаций и распределение классов. Это контроль корректности конвертации из CVAT XML в COCO JSON перед обучением детектора.


In [ ]:
coco_summary = pd.DataFrame({
    "images": [len(coco["images"])],
    "annotations": [len(coco["annotations"])],
    "categories": [len(coco["categories"])],
})
display(coco_summary)

coco_id_to_label = {c["id"]: c["name"] for c in coco["categories"]}
coco_labels = pd.Series([coco_id_to_label[a["category_id"]] for a in coco["annotations"]]).value_counts()
gt_labels = gt_eval_df["label"].value_counts()
class_compare = pd.DataFrame({"GT": gt_labels, "COCO extracted": coco_labels}).fillna(0).astype(int)
display(class_compare)

fig, ax = plt.subplots(figsize=(7, 4))
class_compare.plot(kind="bar", ax=ax)
ax.set_title("Классы: GT vs извлечённая COCO-разметка")
ax.set_xlabel("класс")
ax.set_ylabel("число bbox")
plt.tight_layout()
plt.show()

max_image_id = max(a["image_id"] for a in coco["annotations"]) if coco["annotations"] else -1
pd.DataFrame({
    "проверка": ["max image_id < NUM_FRAMES", "число images == NUM_FRAMES"],
    "результат": [max_image_id < NUM_FRAMES, len(coco["images"]) == NUM_FRAMES],
})


## Визуализация восстановленной разметки

Показаны кадры с наилучшим и наихудшим mean IoU для лучшего OpenCV-метода, а также примеры промежуточных масок предобработки. Красные рамки — GT, зелёные — извлечённая разметка.


In [ ]:
def draw_boxes(image, df, color, label_prefix):
    result = image.copy()
    for _, row in df.iterrows():
        x1, y1, x2, y2 = map(int, [row.x1, row.y1, row.x2, row.y2])
        cv2.rectangle(result, (x1, y1), (x2, y2), color, 2)
        text = f"{label_prefix}:{row.label}"
        cv2.putText(result, text, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return result

def mask_from_diff(input_frame, output_frame):
    diff = cv2.absdiff(input_frame, output_frame)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

def mask_from_hsv(output_frame):
    hsv = cv2.cvtColor(output_frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0, 60, 80]), np.array([179, 255, 255]))
    gray = cv2.cvtColor(output_frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    mask = cv2.bitwise_and(mask, edges)
    return cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=2)

def mask_from_edges(output_frame):
    gray = cv2.cvtColor(output_frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 80, 180)
    return cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)

fm = frame_metrics[best_method].sort_values("mean_iou")
valid_fm = fm[fm["frame_id"].lt(NUM_FRAMES)]
show_frames = valid_fm.tail(2)["frame_id"].astype(int).tolist() + valid_fm.head(2)["frame_id"].astype(int).tolist()
show_frames = list(dict.fromkeys(show_frames))

for frame_id in show_frames[:4]:
    input_frame = cv2.imread(str(input_meta["paths"][frame_id]))
    output_frame = cv2.imread(str(output_meta["paths"][frame_id]))
    row = valid_fm[valid_fm["frame_id"].eq(frame_id)].iloc[0]
    tag = "good" if row["mean_iou"] >= valid_fm["mean_iou"].median() else "bad"
    image = draw_boxes(input_frame, gt_eval_df[gt_eval_df["frame_id"].eq(frame_id)], (0, 0, 255), "gt")
    image = draw_boxes(image, best_pred_df[best_pred_df["frame_id"].eq(frame_id)], (0, 255, 0), "pred")
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("output")
    axes[1].imshow(mask_from_diff(input_frame, output_frame), cmap="gray")
    axes[1].set_title("diff mask")
    axes[2].imshow(mask_from_hsv(output_frame), cmap="gray")
    axes[2].set_title("hsv mask")
    axes[3].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[3].set_title(f"{tag}, frame {frame_id}, IoU={row['mean_iou']:.2f}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## Создание нового видео с восстановленной разметкой

Видео `artifacts/recovered_markup.mp4` содержит только извлеченную нами разметку, отрисованную средствами OpenCV поверх исходного `input.mp4`.

In [ ]:
RECOVERED_VIDEO = ARTIFACTS_DIR / "recovered_markup.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(RECOVERED_VIDEO), fourcc, input_meta["fps"] / FRAME_STRIDE, (input_meta["width"], input_meta["height"]))

for frame_id, path in enumerate(tqdm(input_meta["paths"], desc="render video")):
    image = cv2.imread(str(path))
    image = draw_boxes(image, best_pred_df[best_pred_df["frame_id"].eq(frame_id)], (0, 255, 0), "extracted")
    writer.write(image)

writer.release()
RECOVERED_VIDEO

## Подготовка train/validation split

Детектор обучается на извлеченной COCO-разметке. Валидационная выборка нужна для контроля переобучения, early stopping и расчета mAP.

In [ ]:
frame_ids = sorted(best_pred_df["frame_id"].unique().tolist())
random.shuffle(frame_ids)
val_size = max(1, int(0.2 * len(frame_ids)))
val_ids = set(frame_ids[:val_size])
train_ids = set(frame_ids[val_size:])

train_df = best_pred_df[best_pred_df["frame_id"].isin(train_ids)].copy()
val_df = best_pred_df[best_pred_df["frame_id"].isin(val_ids)].copy()

label_names = sorted(best_pred_df["label"].unique().tolist())
label_to_id = {label: i + 1 for i, label in enumerate(label_names)}
id_to_label = {v: k for k, v in label_to_id.items()}

pd.DataFrame({"split": ["train", "val"], "frames": [len(train_ids), len(val_ids)], "boxes": [len(train_df), len(val_df)]})

## Dataset и DataLoader для torchvision detection

Модель получает изображение и target-словарь с `boxes`, `labels`, `area`, `iscrowd` и `image_id`.

In [ ]:
class CocoLikeDetectionDataset(Dataset):
    def __init__(self, image_paths, annotations, label_to_id, frame_ids):
        self.image_paths = image_paths
        self.annotations = annotations
        self.label_to_id = label_to_id
        self.frame_ids = sorted(list(frame_ids))

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        frame_id = self.frame_ids[idx]
        image = read_image(str(self.image_paths[frame_id])).float() / 255.0
        rows = self.annotations[self.annotations["frame_id"].eq(frame_id)]
        boxes = []
        labels = []
        for _, row in rows.iterrows():
            boxes.append([float(row.x1), float(row.y1), float(row.x2), float(row.y2)])
            labels.append(self.label_to_id[row.label])
        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) else torch.zeros((0,), dtype=torch.float32)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([frame_id], dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = CocoLikeDetectionDataset(input_meta["paths"], train_df, label_to_id, train_ids)
val_dataset = CocoLikeDetectionDataset(input_meta["paths"], val_df, label_to_id, val_ids)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

## Обучение Faster R-CNN

Используется готовая архитектура `fasterrcnn_resnet50_fpn` из `torchvision.models.detection`. Для контроля переобучения применяются валидационная выборка, сохранение лучшей модели и early stopping.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(label_to_id) + 1
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

history = []
best_val_loss = float("inf")
bad_epochs = 0
BEST_MODEL_PATH = MODELS_DIR / "best_fasterrcnn.pth"

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    losses = []
    for images, targets in tqdm(loader, desc="train", leave=False):
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
        loss_dict = model(images, targets)
        loss = sum(value for value in loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0

def validation_loss(model, loader, device):
    model.train()
    losses = []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="val loss", leave=False):
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
            loss_dict = model(images, targets)
            loss = sum(value for value in loss_dict.values())
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validation_loss(model, val_loader, device)
    lr_scheduler.step(val_loss)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": optimizer.param_groups[0]["lr"]})
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bad_epochs = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        bad_epochs += 1
    if bad_epochs >= PATIENCE:
        break

history_df = pd.DataFrame(history)
history_df

## Графики потерь

Расхождение train loss и validation loss используется как индикатор переобучения. Early stopping останавливает обучение, если качество на валидации перестает улучшаться.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
plt.title("Динамика потерь Faster R-CNN")
plt.xlabel("Эпоха")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

## Оценка mAP на валидационной выборке

mAP считается на валидационных кадрах с помощью `torchmetrics.detection.MeanAveragePrecision`.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="mAP"):
        images_device = [image.to(device) for image in images]
        outputs = model(images_device)
        preds = []
        refs = []
        for output, target in zip(outputs, targets):
            preds.append({"boxes": output["boxes"].detach().cpu(), "scores": output["scores"].detach().cpu(), "labels": output["labels"].detach().cpu()})
            refs.append({"boxes": target["boxes"].detach().cpu(), "labels": target["labels"].detach().cpu()})
        metric.update(preds, refs)

map_result = metric.compute()
map_table = pd.DataFrame([{key: float(value) if torch.is_tensor(value) and value.numel() == 1 else str(value) for key, value in map_result.items()}])
map_table

## Визуализация успешных и ошибочных предсказаний модели

На кадрах ниже синим показаны предсказания обученной модели, красным — валидационная разметка. Ошибки обычно связаны с мелкими объектами, перекрытиями и кадрами, где восстановленная разметка была шумной.

In [ ]:
def predict_frame(model, image_path, threshold=0.5):
    model.eval()
    image = read_image(str(image_path)).float() / 255.0
    with torch.no_grad():
        output = model([image.to(device)])[0]
    keep = output["scores"].detach().cpu() >= threshold
    boxes = output["boxes"].detach().cpu()[keep].numpy()
    labels = output["labels"].detach().cpu()[keep].numpy()
    scores = output["scores"].detach().cpu()[keep].numpy()
    rows = []
    for box, label, score in zip(boxes, labels, scores):
        rows.append({"x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3], "label": id_to_label.get(int(label), "object"), "score": float(score)})
    return pd.DataFrame(rows)

val_sample = sorted(list(val_ids))[:6]
for frame_id in val_sample:
    image = cv2.imread(str(input_meta["paths"][frame_id]))
    image = draw_boxes(image, val_df[val_df["frame_id"].eq(frame_id)], (0, 0, 255), "val")
    predicted = predict_frame(model, input_meta["paths"][frame_id], threshold=0.5)
    image = draw_boxes(image, predicted, (255, 0, 0), "model")
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(f"Кадр {frame_id}: предсказания Faster R-CNN")
    plt.axis("off")
    plt.show()

## Интерпретация результатов анализа данных

**Классические методы OpenCV** решают задачу «воровства координат» без обучения: diff опирается на появление новых пикселей, hsv — на цвет рамок, edges — на контуры. На дорожной сцене с фиксированной камерой diff обычно наиболее устойчив.

**Ограничения датасета:** кадры сильно коррелированы по времени, ракурс и освещение почти не меняются, классы могут быть несбалансированы. Это снижает «пригодность» данных для обобщения, но достаточно для лабораторной демонстрации пайплайна.

**Метрики:** IoU (Jaccard) оценивает геометрию bbox; precision / recall / F1 — полноту и точность извлечения. Классы extracted bbox назначаются через GT и не «украдены» с видео.

Далее extracted COCO-разметка используется для обучения Faster R-CNN — уже как отдельный этап нейросетевой детекции.


## Сохранение итоговых таблиц

В артефакты сохраняются сравнительная таблица методов, покадровые метрики лучшего метода, COCO-аннотация, обученная модель и видео с восстановленной разметкой.

In [ ]:
comparison_df.to_csv(ARTIFACTS_DIR / "method_comparison.csv", index=False)
frame_metrics[best_method].to_csv(ARTIFACTS_DIR / "best_method_frame_metrics.csv", index=False)
best_pred_df.to_csv(ARTIFACTS_DIR / "extracted_boxes.csv", index=False)
history_df.to_csv(ARTIFACTS_DIR / "training_history.csv", index=False)
map_table.to_csv(ARTIFACTS_DIR / "map_metrics.csv", index=False)

result_files = sorted([str(path) for path in ARTIFACTS_DIR.glob("*")]) + sorted([str(path) for path in MODELS_DIR.glob("*")])
result_files